# Jupiter to automize clustering for Cats

In [8]:
import pandas as pd
import numpy as np
import sklearn.metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import autosklearn.classification

## I. Open the datasets

In [9]:
df = pd.read_csv('data/OutCatdata.csv', na_filter= False)
df = df.drop("Unnamed: 0", axis= 1)

/tmp/ipykernel_12/3855532020.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/OutCatdata.csv', na_filter= False)


/!\\ the NA values are dropped /!\\

## first look at the data

In [10]:
df

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,Hunt,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


### Split the datasets between classes to predict and data

In [11]:
variables = df.drop(['Hunt'], axis = 1)
variables

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


In [12]:
classes = df['Hunt']
classes

0          Yes
1          Yes
2          Yes
3          Yes
4          Yes
          ... 
1057315    Yes
1057316    Yes
1057317    Yes
1057318    Yes
1057319    Yes
Name: Hunt, Length: 1057320, dtype: object

We have 2 resulting df : 

* classes consisting of the true Hunt status
* variables consisting of the factors

## Création du modèle qualitatif

In [13]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=1000, 
    per_run_time_limit=120, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

*les param minimum pour la tâche* : 

- time_left_for_this_task= 2000 s
- per_run_time_limit=30 cycles
- n_jobs = 16 coeur
- memory_limit = 24 Go

### Génération des jeux de test, validation

pour notre entrainement, nous prenons des proportions de 67% de test et 33% de test

Les méthodes testées sont : 

* Forêt aléatoire
* Latent Dirichlet Allocation
* Multilayered Perceptron
* Baisien naif
* k plus proches voisins 

### Dans un premier temps, nous allons utiliser seulement les données Quantitatives

#### gestion de la suppression des colonnes quantitatives

In [14]:
dfQuali = variables.drop(["event.id","timestamp","location.long","location.lat",
                          "animal.id","StartDate","StartHours","EndDate","EndHours"],
                         axis = 1).astype('category')

Crée le jeu de test qualitatif

In [15]:
variables_trainQ, variables_testQ, classes_trainQ, classes_testQ = train_test_split(
        																dfQuali, classes, test_size = 0.33, random_state=0)

Transformation de classesQ en `category` à la place de `object`

In [16]:
classes_testQ = classes_testQ.astype("category")
classes_trainQ = classes_trainQ.astype("category")

application des paramêtre afin de crée le modèle

In [17]:
cls.fit(variables_trainQ, classes_trainQ)

/usr/local/lib/python3.8/dist-packages/autosklearn/data/target_validator.py:187: UserWarning: Fitting transformer with a pandas series which has the dtype category. Inverse transform may not be able preserve dtype when converting to np.ndarray
  warnings.warn(
Process pynisher function call:
Traceback (most recent call last):
  File "/usr/local/lib/python3.8/dist-packages/sklearn/utils/_encode.py", line 132, in _unique_python
    uniques = sorted(uniques_set)
TypeError: '<' not supported between instances of 'str' and 'int'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.8/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/usr/lib/python3.8/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.8/dist-packages/pynisher/limit_function_call.py", line 133, in subprocess_func
    return_value = ((func(*args, **kwargs

[WARNING] [2025-04-25 09:10:49,892:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 412 not found
[WARNING] [2025-04-25 09:10:49,892:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 102 not found
[WARNING] [2025-04-25 09:10:49,892:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 367 not found
[WARNING] [2025-04-25 09:10:49,893:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 262 not found
[WARNING] [2025-04-25 09:10:49,893:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 37 not found
[WARNING] [2025-04-25 09:10:49,893:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 605 not found
[WARNING] [2025-04-25 09:10:49,893:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 88 not found
[WARNING] [2025-04-25 09:10:49,893:Client-AutoMLSMBO(1)::278cb021-21b5-11f0-800c-d276ce1796e4] Configuration 426 not found
[WARNING] [2025-04

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=120,
                      time_left_for_this_task=1000)

Ne fournis pas de résultats pertinant, potentielement par manqque de temps de calcul

In [18]:
cls.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


Ici, nous voyons que le modèle n'est même pas arriver à faire une ligne

In [19]:
predictions_Hunt = list(cls.predict(variables_testQ))

précision : 

In [20]:
print("Accuracy score:", sklearn.metrics.accuracy_score(np.array(classes_testQ), predictions_Hunt))

Accuracy score: 0.14220041499959876


table des stats

In [21]:
print( sklearn.metrics.classification_report(classes_testQ, predictions_Hunt) )

/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

          NA       0.14      1.00      0.25     49616
          No       0.00      0.00      0.00     49743
         Yes       0.00      0.00      0.00    249557

    accuracy                           0.14    348916
   macro avg       0.05      0.33      0.08    348916
weighted avg       0.02      0.14      0.04    348916



/usr/local/lib/python3.8/dist-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


décevant : on a un précision catastrophique.

Nous allons regarder la matrice de confusion pour potentiellement observer quel groupe est le mieu prédit

In [22]:
np.round( confusion_matrix(classes_testQ, predictions_Hunt), 3)

array([[ 49616,      0,      0],
       [ 49743,      0,      0],
       [249557,      0,      0]])

Le problème est clair : on prédit tout en une classe

## Tentative avec du quantitatif

### Ouverture du csv

In [23]:
dfQuanti = pd.read_csv('data/OutCatdataQuantiNormZ.csv', na_filter= False)
dfQuanti = dfQuanti.drop("Unnamed: 0", axis= 1)

/tmp/ipykernel_12/2793842605.py:1: DtypeWarning: Columns (6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  dfQuanti = pd.read_csv('data/OutCatdataQuantiNormZ.csv', na_filter= False)


### Suppression des lignes ayant des Na

In [24]:
dfQuanti = dfQuanti[~np.any(dfQuanti == "NA",axis=1)]

### Création des sous jeux de données de test et d'entrainement

In [25]:
x = dfQuanti.drop('Hunt', axis=1).to_numpy().astype(np.float64)
y = dfQuanti.Hunt.astype(object)

y[y ==	-2.16472428387967] = "Na"
y[y == 	-0.78922734140023] = "No"
y[y ==   0.586269601079213]  = "Yes"

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size = 0.33, random_state=0)

### Établissement d'un modèle

In [26]:
cls_hunt = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=900, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

Ajustment du modèle a notre jeu de données

In [27]:
cls_hunt.fit(x_train, y_train, dataset_name='Cat Data')

[WARNING] [2025-04-25 09:33:45,137:Client-AutoMLSMBO(1)::Cat Data] Configuration 412 not found
[WARNING] [2025-04-25 09:33:45,138:Client-AutoMLSMBO(1)::Cat Data] Configuration 367 not found
[WARNING] [2025-04-25 09:33:45,138:Client-AutoMLSMBO(1)::Cat Data] Configuration 426 not found
[WARNING] [2025-04-25 09:33:45,138:Client-AutoMLSMBO(1)::Cat Data] Configuration 88 not found
[WARNING] [2025-04-25 09:33:45,138:Client-AutoMLSMBO(1)::Cat Data] Configuration 605 not found
[WARNING] [2025-04-25 09:33:45,138:Client-AutoMLSMBO(1)::Cat Data] Configuration 37 not found
[WARNING] [2025-04-25 09:33:45,139:Client-AutoMLSMBO(1)::Cat Data] Configuration 69 not found
[WARNING] [2025-04-25 09:33:45,139:Client-AutoMLSMBO(1)::Cat Data] Configuration 206 not found
[WARNING] [2025-04-25 09:33:45,139:Client-AutoMLSMBO(1)::Cat Data] Configuration 262 not found
[WARNING] [2025-04-25 09:33:45,139:Client-AutoMLSMBO(1)::Cat Data] Configuration 173 not found
[WARNING] [2025-04-25 09:33:45,139:Client-AutoMLSMBO(

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=100,
                      time_left_for_this_task=900)

## Affichage des résultats

### Affichage des facteurs

In [28]:
cls_hunt.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
5,1,0.04,k_nearest_neighbors,0.000000,41.423518
85,4,0.02,k_nearest_neighbors,0.000000,97.328940
95,3,0.06,k_nearest_neighbors,0.000000,25.023483
97,2,0.02,k_nearest_neighbors,0.000000,85.988688
99,6,0.04,k_nearest_neighbors,0.000000,56.883847
100,11,0.02,k_nearest_neighbors,0.000000,96.171647
103,5,0.06,k_nearest_neighbors,0.000000,57.836018
110,8,0.06,k_nearest_neighbors,0.000000,32.980234
111,7,0.08,k_nearest_neighbors,0.000000,49.757138


### Stockage des prédictions

In [29]:
predictions_Hunt = list(cls_hunt.predict(x_test))

### Affichage des stats

In [30]:
print( sklearn.metrics.classification_report(y_test, predictions_Hunt) )

              precision    recall  f1-score   support

          Na       1.00      1.00      1.00     47640
          No       1.00      1.00      1.00     49890
         Yes       1.00      1.00      1.00    249395

    accuracy                           1.00    346925
   macro avg       1.00      1.00      1.00    346925
weighted avg       1.00      1.00      1.00    346925



### Affichage de la matrice de confusion

In [31]:
np.round( confusion_matrix(y_test, predictions_Hunt), 3)

array([[ 47640,      0,      0],
       [     0,  49890,      0],
       [     0,      0, 249395]])

# Prédiction du nombre de proie par chats

## création des 2 matrices

In [101]:
x2 = dfQuanti.drop('N.pray', axis=1).to_numpy().astype(np.float64)
y2 = dfQuanti['N.pray'].astype(object)

### To-do ya des strings dans dfQuanti['N.pray'] 

on les supprimes en copiant dans une liste temp que les float

In [102]:
out = []
for i in y2 : 
    if type(i) == float:
        out.append(i)
    elif type(i) == str :
        out.append(float(i))

y2 = pd.array(out).astype(object)

## On a fini le formatage, c'est l'heure de faire le jeu de test et le jeu d'entrainement

In [103]:
x2_train, x2_test, y2_train, y2_test = train_test_split(x2, y2,
                                                    test_size = 0.33, random_state=0)

## Configuration du modèle

In [104]:
cls_Npray = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=900, 
    per_run_time_limit=100, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

### Affinage du modèle

In [105]:
cls_Npray.fit(x2_train, y2_train, dataset_name='Cat Data')

ValueError: Classification with data of type unknown is not supported. Supported types are ['binary', 'multiclass', 'multilabel-indicator']. You can find more information about scikit-learn data types in: https://scikit-learn.org/stable/modules/multiclass.html

## Affichage des résultats

### Affichage des facteurs

In [ ]:
cls_Npray.leaderboard()

### Stockage des prédictions

In [ ]:
predictions_Npray = list(cls_Npray.predict(x2_test))

### Affichage des stats

In [ ]:
print( sklearn.metrics.classification_report(y2_test, predictions_Npray) )

### Affichage de la matrice de confusion

In [ ]:
np.round( confusion_matrix(y2_test, predictions_Npray), 3)